# Tenant Isolation with Prompt Caching

This notebook demonstrates how to implement per-tenant cache isolation using a SHA-256 hash prefix pattern. Each tenant's cached content is keyed separately, preventing cross-tenant cache sharing.

## What you'll learn

- SHA-256 hash prefix pattern for tenant-specific cache keys
- Verifying cache isolation: tenant A's cache is not read by tenant B
- Verifying cache sharing within a tenant: same tenant, different queries reuse the cache

## How it works

A SHA-256 hash of the `tenant_id` is prepended to the cached content. Since the hash changes the content prefix, Bedrock creates a separate cache entry for each tenant.

```python
sha256_hash = hashlib.sha256(tenant_id.encode()).hexdigest()
instructions = f"{sha256_hash}:{instructions}"
```

## Prerequisites

- AWS credentials configured (default profile)
- Access to Claude Sonnet 4.6 on Amazon Bedrock
- `boto3 >= 1.35.76`

In [ ]:
!pip install --upgrade --quiet boto3

In [ ]:
import boto3
import json
import time
import hashlib

MODEL_ID = "global.anthropic.claude-sonnet-4-6-v1:0"
AWS_REGION = "us-west-2"

bedrock = boto3.client("bedrock-runtime", region_name=AWS_REGION)
print(f"Model: {MODEL_ID}")

## Sample Document and Instructions

A long instruction prompt and an inline document provide enough tokens to exceed the 1,024-token cache threshold.

In [ ]:
INSTRUCTIONS = (
    "I will provide you with a document, followed by a question about its content. "
    "Your task is to analyze the document, extract relevant information, and provide "
    "a comprehensive answer to the question. Please follow these detailed instructions:"
    "\n\n1. Identifying Relevant Quotes:"
    "\n   - Carefully read through the entire document."
    "\n   - Identify sections of the text that are directly relevant to answering the question."
    "\n   - Select quotes that provide key information, context, or support for the answer."
    "\n   - Quotes should be concise and to the point, typically no more than 2-3 sentences each."
    "\n   - Choose a diverse range of quotes if multiple aspects of the question need to be addressed."
    "\n   - Aim to select between 2 to 5 quotes, depending on the complexity of the question."
    "\n\n2. Presenting the Quotes:"
    "\n   - List the selected quotes under the heading 'Relevant quotes:'"
    "\n   - Number each quote sequentially, starting from [1]."
    "\n   - Present each quote exactly as it appears in the original text, enclosed in quotation marks."
    "\n   - If no relevant quotes can be found, write 'No relevant quotes' instead."
    "\n\n3. Formulating the Answer:"
    "\n   - Begin your answer with the heading 'Answer:' on a new line after the quotes."
    "\n   - Provide a clear, concise, and accurate answer to the question based on the information in the document."
    "\n   - Ensure your answer is comprehensive and addresses all aspects of the question."
    "\n   - Use information from the quotes to support your answer, but do not repeat them verbatim."
    "\n   - Maintain a logical flow and structure in your response."
    "\n\n4. Referencing Quotes in the Answer:"
    "\n   - Do not explicitly mention or introduce quotes in your answer."
    "\n   - Instead, add the bracketed number of the relevant quote at the end of each sentence."
    "\n   - If a sentence is supported by multiple quotes, include all relevant quote numbers."
    "\n\n5. Handling Uncertainty:"
    "\n   - If the document does not contain enough information, clearly state this."
    "\n   - Provide any partial information that is available."
    "\n\n6. Maintaining Objectivity:"
    "\n   - Stick to the facts presented in the document."
    "\n   - Do not include personal opinions or external information."
    "\n\n7. Formatting:"
    "\n   - Use clear paragraph breaks to separate different points."
    "\n   - Employ bullet points or numbered lists if it helps organize information."
    "\n   - Ensure proper grammar, punctuation, and spelling."
    "\n\n8. Length and Depth:"
    "\n   - Provide an answer that is sufficiently detailed to address the question comprehensively."
    "\n   - Avoid unnecessary verbosity. Aim for clarity and conciseness."
    "\n\n9. Dealing with Complex Questions:"
    "\n   - For questions with multiple parts, address each part separately."
    "\n   - Use subheadings or numbered points to break down your answer."
    "\n\n10. Concluding the Answer:"
    "\n    - If appropriate, provide a brief conclusion summarizing the key points."
)

DOCUMENT = """
Prompt caching is a feature in Amazon Bedrock that stores portions of conversation context for reuse.
When a cache checkpoint is hit on subsequent requests, the model skips reprocessing the cached tokens,
resulting in lower Time-To-First-Token (TTFT) and reduced input token costs.

Supported models include Anthropic Claude (Sonnet 4.6, Opus 4.6, Haiku 4.5), Amazon Nova, and select
open-source models. Each model family has a minimum token threshold per cache checkpoint — for example,
Claude Sonnet 4.6 requires at least 1,024 tokens.

Cache entries are scoped per account and region. They expire based on the TTL specified in the request
(default: 5 minutes, maximum: 1 hour for Converse API). The Converse API uses a cachePoint content
block, while the InvokeModel API uses cache_control inside content blocks.

Prompt caching is useful for chat-with-document workflows, coding assistants, agentic workflows with
complex tool definitions, and few-shot learning scenarios with many examples.
"""

QUESTIONS = [
    "What is prompt caching?",
    "What are the use cases for prompt caching?",
]

## Chat Function with Tenant Isolation

In [ ]:
def chat_with_document(document, user_query, tenant_id):
    """Chat with a document using tenant-isolated prompt caching."""
    instructions = INSTRUCTIONS

    if tenant_id:
        sha256_hash = hashlib.sha256(tenant_id.encode()).hexdigest()
        instructions = f"{sha256_hash}:{instructions}"

    document_content = f"Here is the document: <document>{document}</document>"

    response = bedrock.converse(
        modelId=MODEL_ID,
        messages=[{
            "role": "user",
            "content": [
                {"text": instructions},
                {"text": document_content},
                {"cachePoint": {"type": "default"}},
                {"text": user_query}
            ]
        }],
        inferenceConfig={"maxTokens": 500, "temperature": 0, "topP": 1}
    )

    usage = response["usage"]
    text = response["output"]["message"]["content"][0]["text"]
    return usage, text

## Tenant 1 — First Request (Cache Write)

In [ ]:
usage, text = chat_with_document(DOCUMENT, QUESTIONS[0], tenant_id="tenant1")
print("Tenant 1, Request 1 (cache write expected):")
print(json.dumps(usage, indent=2))
print(f"\nResponse preview: {text[:200]}...")

## Tenant 1 — Second Request (Cache Read)

Same tenant, different question. The cached content (instructions + document) should be read from cache.

In [ ]:
usage, text = chat_with_document(DOCUMENT, QUESTIONS[1], tenant_id="tenant1")
print("Tenant 1, Request 2 (cache read expected):")
print(json.dumps(usage, indent=2))

## Tenant 2 — First Request (Cache Write)

Different tenant. The SHA-256 hash prefix differs, so this creates a **separate** cache entry.

In [ ]:
usage, text = chat_with_document(DOCUMENT, QUESTIONS[0], tenant_id="tenant2")
print("Tenant 2, Request 1 (cache write expected — separate from tenant1):")
print(json.dumps(usage, indent=2))

## Tenant 2 — Second Request (Cache Read)

Same tenant 2, different question. Should read from tenant 2's cache.

In [ ]:
usage, text = chat_with_document(DOCUMENT, QUESTIONS[1], tenant_id="tenant2")
print("Tenant 2, Request 2 (cache read expected):")
print(json.dumps(usage, indent=2))

## Verification Summary

Expected behavior across the 4 requests:

| Request | Tenant | Expected Cache Behavior |
|---|---|---|
| 1 | tenant1 | `cacheWriteInputTokens > 0` (new cache entry) |
| 2 | tenant1 | `cacheReadInputTokens > 0` (reuses tenant1 cache) |
| 3 | tenant2 | `cacheWriteInputTokens > 0` (new cache entry — different hash prefix) |
| 4 | tenant2 | `cacheReadInputTokens > 0` (reuses tenant2 cache) |

## Conclusion

The SHA-256 hash prefix pattern provides lightweight tenant isolation for prompt caching:
- No server-side configuration needed — isolation is achieved purely through content prefixing
- Each tenant's cache is independent
- The hash prefix adds minimal token overhead

For more details, see the [Amazon Bedrock Prompt Caching documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html).